# Notebook 5: Comparing Embeddings Across Genomic Language Models

**BMI 511 — Spring 2026**
**Instructor:** Pratik Dutta, Ph.D. | Department of Biomedical Informatics, Stony Brook University

---

## Learning objectives

1. Load **four different genomic language models** and extract embeddings from each.
2. Compare how different architectures, tokenization strategies, and training data shape the embedding space.
3. Visualize all four models side-by-side with UMAP and t-SNE.
4. Quantify separation quality using **silhouette score** and **linear-probe accuracy**.
5. Develop intuition for **which model to choose** for a given downstream task.

## Models we will compare

| Model | Architecture | Tokenization | Parameters | Training data | Max length |
| --- | --- | --- | --- | --- | --- |
| **DNABERT** (Ji et al., 2021) | BERT, 12-layer | 6-mer overlapping | ~110 M | Human genome | 512 tokens |
| **DNABERT-2** (Zhou et al., 2024) | BERT, 12-layer | BPE (learned) | 117 M | Multi-species | 512 tokens |
| **Nucleotide Transformer v2** (Dalla-Torre et al., 2024) | ESM-style, 12-layer | 6-mer non-overlapping | 100 M | 850+ species | 1,000 tokens |
| **HyenaDNA** (Nguyen et al., 2023) | Hyena (SSM) | Character-level | ~7 M | Human genome | 32 kb |

These four models span the major design axes in the field: **transformer vs state-space model**, **k-mer vs BPE vs character tokenization**, **single-species vs multi-species training**, and **small vs large parameter count**.

> **Runtime: T4 GPU required.** We load one model at a time and free GPU memory between models to stay within Colab's 15 GB VRAM.

## 1. Setup

In [ ]:
!pip install -q transformers==4.44.2 datasets einops biopython scikit-learn umap-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import time

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Build a dataset with strong compositional contrast

We create four sequence classes with distinct biological signatures so that differences in how each model represents them become clearly visible.

| Class | Composition | Biological analog |
| --- | --- | --- |
| **CpG-island** | ~80% GC, CG dinucleotide enriched | Promoter CpG islands |
| **TATA-promoter** | ~75% AT, TATAAA motif implanted | Core promoter with TATA box |
| **Poly-A signal** | Mixed, AATAAA motif near 3' end | 3' UTR polyadenylation signal |
| **Random** | Uniform 25% each nucleotide | Intergenic background |

In [ ]:
rng = np.random.default_rng(42)
N_PER_CLASS = 80
SEQ_LEN = 300

def sample_seq(probs, length=SEQ_LEN):
    return ''.join(rng.choice(list("ACGT"), size=length, p=probs))

def implant_motif(seq, motif, region="middle"):
    L = len(seq)
    if region == "middle":
        pos = rng.integers(L // 3, 2 * L // 3)
    elif region == "end":
        pos = rng.integers(L - 40, L - len(motif))
    else:
        pos = rng.integers(0, L - len(motif))
    return seq[:pos] + motif + seq[pos + len(motif):]

sequences, labels = [], []

# Class 0: CpG-island-like (very GC-rich)
for _ in range(N_PER_CLASS):
    seq = sample_seq([0.10, 0.40, 0.40, 0.10])
    for _ in range(5):
        pos = rng.integers(0, SEQ_LEN - 2)
        seq = seq[:pos] + "CG" + seq[pos+2:]
    sequences.append(seq); labels.append(0)

# Class 1: TATA-promoter (AT-rich, TATAAA motif)
for _ in range(N_PER_CLASS):
    seq = sample_seq([0.38, 0.12, 0.12, 0.38])
    seq = implant_motif(seq, "TATAAAGC", region="middle")
    sequences.append(seq); labels.append(1)

# Class 2: Poly-A signal region (mixed, AATAAA near end)
for _ in range(N_PER_CLASS):
    seq = sample_seq([0.30, 0.20, 0.20, 0.30])
    seq = implant_motif(seq, "AATAAA", region="end")
    sequences.append(seq); labels.append(2)

# Class 3: Random intergenic
for _ in range(N_PER_CLASS):
    seq = sample_seq([0.25, 0.25, 0.25, 0.25])
    sequences.append(seq); labels.append(3)

labels = np.array(labels)
LABEL_NAMES = {0: "CpG-island", 1: "TATA-promoter", 2: "PolyA-signal", 3: "Random"}
COLORS = {0: "#1f77b4", 1: "#ff7f0e", 2: "#2ca02c", 3: "#d62728"}

print(f"Total sequences: {len(sequences)}")
for k, name in LABEL_NAMES.items():
    subset = [s for s, l in zip(sequences, labels) if l == k]
    gc = np.mean([(s.count('G') + s.count('C')) / len(s) for s in subset])
    print(f"  class {k} ({name:15s})  n={len(subset):3d}  mean GC = {gc:.2f}")

## 3. Universal embedding helper

We write a single function that works for any HuggingFace model. Each model has different tokenization, so we pass in the model-specific tokenizer. We use **mean-pooling** over non-padded tokens for all models to keep the comparison fair.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def embed_all(seqs, model, tokenizer, device, batch_size=16, max_length=512):
    """Extract mean-pooled embeddings for a list of DNA sequences."""
    all_vecs = []
    for i in tqdm(range(0, len(seqs), batch_size), desc="Embedding", leave=False):
        batch = seqs[i:i+batch_size]
        enc = tokenizer(
            batch, return_tensors="pt",
            padding=True, truncation=True,
            max_length=max_length,
        ).to(device)

        out = model(**enc)
        hidden = out[0] if isinstance(out, tuple) else out.last_hidden_state

        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        all_vecs.append(pooled.cpu().numpy())

    return np.concatenate(all_vecs, axis=0)

def free_gpu():
    """Release GPU memory between models."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"  GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

## 4. Load each model, extract embeddings, then unload

We process one model at a time to stay within Colab's T4 memory. After extracting embeddings, we delete the model and free GPU memory before loading the next one.

### 4.1 DNABERT (original, 6-mer tokenization)

The original DNABERT (Ji et al., *Bioinformatics* 2021) was the first transformer-based genomic language model. It uses **overlapping 6-mers** as tokens: the raw sequence must be converted to space-separated k-mer strings before tokenization.

Key design choice: overlapping k-mers mean `ATCGATCG` becomes `ATCGAT TCGATC CGATCG`. This explicitly encodes local context but inflates the token count.

In [ ]:
from transformers import AutoTokenizer, BertModel, BertConfig

print("=" * 60)
print("MODEL 1: DNABERT (6-mer)")
print("=" * 60)

DNABERT_NAME = "zhihan1996/DNA_bert_6"

tok_dnabert = AutoTokenizer.from_pretrained(DNABERT_NAME)
config_dnabert = BertConfig.from_pretrained(DNABERT_NAME)
model_dnabert = BertModel.from_pretrained(
    DNABERT_NAME, config=config_dnabert
).to(device).eval()

print(f"  Parameters : {sum(p.numel() for p in model_dnabert.parameters())/1e6:.0f} M")
print(f"  Hidden dim : {model_dnabert.config.hidden_size}")
print(f"  Layers     : {model_dnabert.config.num_hidden_layers}")

# DNABERT expects k-mer input: "ATCGATCG" -> "ATCGAT TCGATC CGATCG"
def to_kmer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq) - k + 1)])

seqs_kmer = [to_kmer(s) for s in sequences]

t0 = time.time()
emb_dnabert = embed_all(seqs_kmer, model_dnabert, tok_dnabert, device, max_length=512)
t_dnabert = time.time() - t0
print(f"  Embeddings : {emb_dnabert.shape}  ({t_dnabert:.1f}s)")

del model_dnabert; free_gpu()

### 4.2 DNABERT-2 (BPE tokenization)

DNABERT-2 (Zhou et al., *ICLR* 2024) replaced overlapping k-mers with **Byte-Pair Encoding** (BPE), the same tokenization strategy used by GPT and LLaMA. BPE learns the most frequent substrings from the training corpus and merges them into tokens of variable length. No k-mer preprocessing needed: you feed raw DNA directly.

In [ ]:
print("=" * 60)
print("MODEL 2: DNABERT-2 (BPE)")
print("=" * 60)

DNABERT2_NAME = "zhihan1996/DNABERT-2-117M"

tok_dnabert2 = AutoTokenizer.from_pretrained(DNABERT2_NAME, trust_remote_code=True)
config_dnabert2 = BertConfig.from_pretrained(DNABERT2_NAME)
model_dnabert2 = BertModel.from_pretrained(
    DNABERT2_NAME, config=config_dnabert2
).to(device).eval()

print(f"  Parameters : {sum(p.numel() for p in model_dnabert2.parameters())/1e6:.0f} M")
print(f"  Hidden dim : {model_dnabert2.config.hidden_size}")

t0 = time.time()
emb_dnabert2 = embed_all(sequences, model_dnabert2, tok_dnabert2, device, max_length=512)
t_dnabert2 = time.time() - t0
print(f"  Embeddings : {emb_dnabert2.shape}  ({t_dnabert2:.1f}s)")

del model_dnabert2; free_gpu()

### 4.3 Nucleotide Transformer v2 (100M, multi-species)

The Nucleotide Transformer (Dalla-Torre et al., *Nature Methods* 2024) is trained on **850+ species** with a 6-mer **non-overlapping** tokenizer and an ESM-inspired architecture. We use the 100M parameter variant.

Key differences from DNABERT: non-overlapping 6-mers (so `ATCGATCG` becomes `ATCGAT CG`, not sliding windows), and multi-species training data, which should produce more generalizable representations.

In [ ]:
from transformers import AutoModelForMaskedLM

print("=" * 60)
print("MODEL 3: Nucleotide Transformer v2 (100M)")
print("=" * 60)

NT_NAME = "InstaDeepAI/nucleotide-transformer-v2-100m-multi-species"

tok_nt = AutoTokenizer.from_pretrained(NT_NAME, trust_remote_code=True)

# NT is distributed as a MaskedLM model; we extract the last hidden state
model_nt = AutoModelForMaskedLM.from_pretrained(
    NT_NAME, trust_remote_code=True
).to(device).eval()

print(f"  Parameters : {sum(p.numel() for p in model_nt.parameters())/1e6:.0f} M")

@torch.no_grad()
def embed_nt(seqs, model, tokenizer, device, batch_size=16, max_length=1000):
    """NT-specific: extract last hidden state via output_hidden_states."""
    all_vecs = []
    for i in tqdm(range(0, len(seqs), batch_size), desc="Embedding NT", leave=False):
        batch = seqs[i:i+batch_size]
        enc = tokenizer(
            batch, return_tensors="pt",
            padding=True, truncation=True,
            max_length=max_length,
        ).to(device)
        out = model(**enc, output_hidden_states=True)
        hidden = out.hidden_states[-1]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        all_vecs.append(pooled.cpu().numpy())
    return np.concatenate(all_vecs, axis=0)

t0 = time.time()
emb_nt = embed_nt(sequences, model_nt, tok_nt, device, max_length=1000)
t_nt = time.time() - t0
print(f"  Embeddings : {emb_nt.shape}  ({t_nt:.1f}s)")

del model_nt; free_gpu()

### 4.4 HyenaDNA (character-level, long-range SSM)

HyenaDNA (Nguyen et al., *NeurIPS* 2023) is architecturally the most different model here. Instead of a transformer, it uses **Hyena operators** — a state-space model (SSM) that achieves near-linear scaling with sequence length. It tokenizes DNA at the **character level** (one token per nucleotide) and can handle sequences up to 1 million bp.

Despite having only ~7M parameters (15x smaller than DNABERT-2), the long-range SSM architecture may still capture meaningful patterns.

In [ ]:
print("=" * 60)
print("MODEL 4: HyenaDNA (character-level, SSM)")
print("=" * 60)

HYENA_NAME = "LongSafari/hyenadna-small-32k-seqlen"

# HyenaDNA uses its own tokenizer and model classes via trust_remote_code
from transformers import AutoModelForCausalLM

tok_hyena = AutoTokenizer.from_pretrained(HYENA_NAME, trust_remote_code=True)
model_hyena = AutoModelForCausalLM.from_pretrained(
    HYENA_NAME, trust_remote_code=True,
    torch_dtype=torch.float32,
).to(device).eval()

n_params = sum(p.numel() for p in model_hyena.parameters())
print(f"  Parameters : {n_params/1e6:.1f} M")

@torch.no_grad()
def embed_hyena(seqs, model, tokenizer, device, batch_size=16, max_length=512):
    """HyenaDNA-specific: causal model, extract hidden states and mean-pool."""
    all_vecs = []
    for i in tqdm(range(0, len(seqs), batch_size), desc="Embedding HyenaDNA", leave=False):
        batch = seqs[i:i+batch_size]
        enc = tokenizer(
            batch, return_tensors="pt",
            padding=True, truncation=True,
            max_length=max_length,
        ).to(device)
        out = model(**enc, output_hidden_states=True)
        # Use the last hidden state from the SSM backbone
        hidden = out.hidden_states[-1]   # (B, T, D)
        # Use attention_mask if available, otherwise use input_ids != pad_token_id
        if "attention_mask" in enc:
            mask = enc["attention_mask"].unsqueeze(-1).float()
        else:
            pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
            mask = (enc["input_ids"] != pad_id).unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        all_vecs.append(pooled.cpu().numpy())
    return np.concatenate(all_vecs, axis=0)

t0 = time.time()
emb_hyena = embed_hyena(sequences, model_hyena, tok_hyena, device, max_length=512)
t_hyena = time.time() - t0
print(f"  Embeddings : {emb_hyena.shape}  ({t_hyena:.1f}s)")

del model_hyena; free_gpu()

In [ ]:
# Collect all embeddings
all_embeddings = {
    "DNABERT (6-mer)":       emb_dnabert,
    "DNABERT-2 (BPE)":      emb_dnabert2,
    "NT v2 (100M)":         emb_nt,
    "HyenaDNA (char-SSM)":  emb_hyena,
}

print("All embeddings collected:")
print(f"  {'Model':30s}  {'Shape':15s}  {'Time':>6s}")
print("  " + "-" * 55)
times = {"DNABERT (6-mer)": t_dnabert, "DNABERT-2 (BPE)": t_dnabert2,
         "NT v2 (100M)": t_nt, "HyenaDNA (char-SSM)": t_hyena}
for name, emb in all_embeddings.items():
    print(f"  {name:30s}  {str(emb.shape):15s}  {times[name]:5.1f}s")

## 5. Side-by-side UMAP projections

We project each model's embeddings to 2D using UMAP and plot them in a 2x2 grid. If a model captures the compositional and motif differences, we expect to see four distinct clusters.

In [ ]:
import umap

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (model_name, emb) in zip(axes.flatten(), all_embeddings.items()):
    X = StandardScaler().fit_transform(emb)
    proj = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X)

    for k, cname in LABEL_NAMES.items():
        mask = labels == k
        ax.scatter(proj[mask, 0], proj[mask, 1], s=25, alpha=0.7,
                   label=cname, color=COLORS[k], edgecolor="white", linewidth=0.3)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    ax.legend(fontsize=8, loc="best")

plt.suptitle("UMAP of sequence embeddings across four genomic language models",
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 6. t-SNE comparison

t-SNE emphasizes local neighborhood structure. It may reveal cluster substructure that UMAP smooths over.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (model_name, emb) in zip(axes.flatten(), all_embeddings.items()):
    X = StandardScaler().fit_transform(emb)
    proj = TSNE(n_components=2, perplexity=20, random_state=42, init="pca").fit_transform(X)

    for k, cname in LABEL_NAMES.items():
        mask = labels == k
        ax.scatter(proj[mask, 0], proj[mask, 1], s=25, alpha=0.7,
                   label=cname, color=COLORS[k], edgecolor="white", linewidth=0.3)
    ax.set_title(model_name, fontsize=13, fontweight="bold")
    ax.set_xlabel("t-SNE-1"); ax.set_ylabel("t-SNE-2")
    ax.legend(fontsize=8, loc="best")

plt.suptitle("t-SNE of sequence embeddings across four genomic language models",
             fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 7. Quantitative comparison

Visualization is useful but subjective. Let's measure embedding quality with two standard metrics:

1. **Silhouette score** (unsupervised): how well-separated are the clusters in the full high-dimensional space? Ranges from -1 (bad) to +1 (perfect).

2. **Linear-probe accuracy** (supervised): train a logistic regression on frozen embeddings with 5-fold cross-validation. A high score means the embedding space is linearly separable.

In [ ]:
results = []

for model_name, emb in all_embeddings.items():
    X = StandardScaler().fit_transform(emb)

    sil = silhouette_score(X, labels)

    clf = LogisticRegression(max_iter=2000, C=1.0, multi_class="multinomial")
    cv_scores = cross_val_score(clf, X, labels, cv=5, scoring="accuracy")

    results.append({
        "Model": model_name,
        "Hidden dim": emb.shape[1],
        "Silhouette": round(sil, 3),
        "Probe acc (mean)": round(cv_scores.mean(), 3),
        "Probe acc (std)": round(cv_scores.std(), 3),
    })
    print(f"{model_name:30s}  silhouette={sil:.3f}  "
          f"probe_acc={cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

df_results = pd.DataFrame(results)
df_results

## 8. Visualize the metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

# Silhouette
ax = axes[0]
bars = ax.barh(df_results["Model"], df_results["Silhouette"], color=bar_colors)
ax.set_xlabel("Silhouette score")
ax.set_title("Cluster separation (unsupervised)")
ax.set_xlim(0, max(df_results["Silhouette"].max() * 1.3, 0.3))
for bar, val in zip(bars, df_results["Silhouette"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=11)

# Linear probe
ax = axes[1]
bars = ax.barh(df_results["Model"], df_results["Probe acc (mean)"],
               xerr=df_results["Probe acc (std)"],
               color=bar_colors, capsize=4)
ax.set_xlabel("5-fold CV accuracy")
ax.set_title("Linear-probe classification (supervised)")
ax.set_xlim(0, 1.15)
for bar, val in zip(bars, df_results["Probe acc (mean)"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=11)

plt.tight_layout()
plt.show()

## 9. Pairwise cosine similarity heatmaps

We sample 8 sequences per class and plot cosine similarity. The colormap is scaled to the **actual data range** for each model so that the block-diagonal structure (high within-class similarity) is visible.

In [ ]:
sample_idx = np.concatenate([np.where(labels == k)[0][:8] for k in range(4)])
sub_labels = labels[sample_idx]
tick_labels = [f"{LABEL_NAMES[l][:5]}.{i%8}" for i, l in enumerate(sub_labels)]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, (model_name, emb) in zip(axes.flatten(), all_embeddings.items()):
    sub_emb = emb[sample_idx]
    sim = cosine_similarity(sub_emb)

    # Scale colormap to data range so structure is visible
    off_diag = sim[np.triu_indices_from(sim, k=1)]
    vmin = np.percentile(off_diag, 2)
    vmax = np.percentile(off_diag, 98)

    sns.heatmap(sim, ax=ax, xticklabels=tick_labels, yticklabels=tick_labels,
                cmap="RdBu_r", vmin=vmin, vmax=vmax,
                cbar_kws={"label": "cosine similarity"})
    ax.set_title(f"{model_name}\n(range: {vmin:.3f} to {vmax:.3f})", fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", rotation=60, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

plt.suptitle("Pairwise cosine similarity (colormap scaled per model)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. Tokenization comparison

The same DNA sequence produces very different tokens depending on the model. This directly affects what patterns each model can learn in a single forward pass.

In [ ]:
sample = sequences[0][:60]
print(f"Raw DNA ({len(sample)} bp):\n{sample}\n")
print(f"{'Model':30s}  {'n_tokens':>8s}  first tokens")
print("-" * 90)

# DNABERT (6-mer)
ids = tok_dnabert(to_kmer(sample), return_tensors="pt")["input_ids"][0]
tokens = [t for t in tok_dnabert.convert_ids_to_tokens(ids) if t not in ("[CLS]", "[SEP]", "[PAD]")]
print(f"{'DNABERT (6-mer)':30s}  {len(tokens):8d}  {tokens[:6]}...")

# DNABERT-2 (BPE)
ids = tok_dnabert2(sample, return_tensors="pt")["input_ids"][0]
tokens = [t for t in tok_dnabert2.convert_ids_to_tokens(ids) if t not in ("[CLS]", "[SEP]", "<s>", "</s>")]
print(f"{'DNABERT-2 (BPE)':30s}  {len(tokens):8d}  {tokens[:6]}...")
print(f"{'  token lengths (bp)':30s}  {'':8s}  {[len(t) for t in tokens[:6]]}")

# NT v2
ids = tok_nt(sample, return_tensors="pt")["input_ids"][0]
tokens = [t for t in tok_nt.convert_ids_to_tokens(ids) if t not in ("[CLS]", "[SEP]", "<s>", "</s>", "<pad>")]
print(f"{'NT v2 (6-mer non-overlap)':30s}  {len(tokens):8d}  {tokens[:6]}...")

# HyenaDNA
ids = tok_hyena(sample, return_tensors="pt")["input_ids"][0]
tokens = [t for t in tok_hyena.convert_ids_to_tokens(ids)][:12]
print(f"{'HyenaDNA (character)':30s}  {len(sample):8d}  {tokens[:12]}...")

**Key observations:**

* **DNABERT** produces the most tokens (overlapping 6-mers), which means the model processes many overlapping views of the same region.
* **DNABERT-2** compresses the input via BPE; each token covers a variable number of base pairs.
* **NT v2** uses non-overlapping 6-mers, giving the fewest tokens of the transformer models.
* **HyenaDNA** uses one token per nucleotide (character-level), but its SSM architecture handles this efficiently with near-linear scaling.

## 11. Embedding space geometry: PCA variance analysis

How many dimensions carry useful information? We plot the cumulative variance explained by the top PCA components for each model. A model that concentrates information in fewer components has a more structured, lower-dimensional embedding space.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for model_name, emb in all_embeddings.items():
    X = StandardScaler().fit_transform(emb)
    n_comp = min(50, emb.shape[1])
    pca = PCA(n_components=n_comp, random_state=0).fit(X)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    ax.plot(range(1, n_comp + 1), cumvar, linewidth=2, label=model_name)

ax.axhline(y=0.9, color="gray", linestyle="--", alpha=0.5)
ax.text(2, 0.91, "90% threshold", fontsize=9, color="gray")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("How many dimensions carry the information?")
ax.legend(fontsize=10)
ax.set_xlim(1, 50)
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Components needed for 90% variance:")
for model_name, emb in all_embeddings.items():
    X = StandardScaler().fit_transform(emb)
    pca = PCA(n_components=min(emb.shape[0], emb.shape[1]), random_state=0).fit(X)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n90 = int(np.searchsorted(cumvar, 0.9)) + 1
    print(f"  {model_name:30s}  {n90} components")

## 12. Discussion: choosing the right model

| Model | Best for | Limitation |
| --- | --- | --- |
| **DNABERT** | Short regulatory sequences (<500 bp); explainability via k-mer attention | Human-genome only; overlapping k-mers inflate token count |
| **DNABERT-2** | General-purpose; strong baseline on GUE benchmark; efficient BPE | Max 512 BPE tokens (~2 kb); not multi-species |
| **NT v2** | Multi-species tasks; metagenomics; comparative genomics | 6-mer tokenization less flexible than BPE |
| **HyenaDNA** | Ultra-long sequences (32k+); low-resource settings (only 7M params) | Causal model (unidirectional); smaller capacity |

**Rules of thumb for your research:**
* **Default choice**: DNABERT-2 (BPE is efficient, well-benchmarked, easy to fine-tune).
* **Multi-species / metagenomics**: Nucleotide Transformer.
* **Very long regions** (whole genes, TADs, enhancer-promoter loops): HyenaDNA or Caduceus.
* **Maximum scale**: Evo 2 (40B parameters, 1M token context) when compute is not a constraint.

## 13. Beyond Colab: the Evo model family

Two models are too large for a free Colab T4 but represent the frontier of the field:

### Evo (Nguyen et al., *Science* 2024)
* **7B parameters**, StripedHyena architecture (SSM + attention hybrid)
* Character-level tokenization, 131k context
* Trained on OpenGenome: ~300B tokens from prokaryotic genomes
* **Autoregressive** (generative): can generate novel DNA sequences, not just encode
* First gLM shown to design functional CRISPR systems and gene regulatory elements

### Evo 2 (Brixi et al., *Nature* 2026)
* **1B / 7B / 20B / 40B** parameter family
* Trained on OpenGenome2: **9.3 trillion base pairs** spanning all domains of life
* 1 million token context window at single-nucleotide resolution
* Zero-shot variant effect prediction (BRCA1 pathogenic variants) without any fine-tuning
* First gLM at the same parameter scale as frontier text LLMs

Both Evo models use the **StripedHyena** architecture rather than a standard transformer, representing a convergence between the SSM approach (like HyenaDNA) and attention mechanisms.

**Hardware requirements:**
* Evo 1 (7B): fits on a single A100 (80GB) in bfloat16
* Evo 2 1B base: requires Hopper GPU (H100/H200) with FP8 via Transformer Engine
* Evo 2 7B: A100 in bfloat16, or H100 with FP8
* Evo 2 40B: multiple GPUs

## 14. Exercises

### In-class (Colab T4)

1. Change the pooling strategy from **mean** to **CLS** (first token) for DNABERT and DNABERT-2. Does the silhouette score improve or degrade?
2. Add a **fifth sequence class** (e.g. splice-donor-like: implant `GTAAGT` at a random position in uniform background). Re-run the full comparison. Which model separates 5 classes best?
3. Increase `SEQ_LEN` to 1000. Does HyenaDNA benefit more than the transformer models from longer context?

### Take-home (requires A100 GPU, e.g. Colab Pro)

4. **Evo challenge.** Install Evo 1 and extract embeddings for our 320 sequences:
```python
!pip install evo-model
from transformers import AutoConfig, AutoModelForCausalLM

model_name = "togethercomputer/evo-1-8k-base"
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True, revision="1.1_fix")
config.use_cache = True
model = AutoModelForCausalLM.from_pretrained(
    model_name, config=config, trust_remote_code=True, revision="1.1_fix"
)
```
   Compare its silhouette score and linear-probe accuracy against the four models from this notebook. Does a 7B autoregressive model outperform the smaller bidirectional encoders on this task?

5. **Evo 2 challenge.** If you have access to an H100, install Evo 2 (`pip install evo2`) and extract intermediate embeddings (the paper recommends layer 26 for the 7B model):
```python
from evo2 import Evo2
evo2_model = Evo2("evo2_7b")
outputs, embeddings = evo2_model(
    input_ids, return_embeddings=True, layer_names=["blocks.28.mlp.l3"]
)
```
   Does the 100x larger model produce meaningfully better cluster separation on this simple task?

6. **Real-data challenge.** Replace the synthetic dataset with the GUE benchmark promoter task:
```python
from datasets import load_dataset
ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", "promoter_all")
```
   Run all four models and report which achieves the highest linear-probe accuracy on real promoter sequences.

## 15. Recap

* Different gLMs produce **qualitatively different** embedding spaces for the same sequences.
* **Tokenization** (overlapping k-mer vs BPE vs non-overlapping k-mer vs character) is a major source of these differences.
* **Silhouette score** and **linear-probe accuracy** are fast, quantitative ways to compare model quality without task-specific fine-tuning.
* The transformer models (DNABERT, DNABERT-2, NT v2) are bidirectional encoders; HyenaDNA is a unidirectional SSM. Both families produce useful embeddings.
* For cutting-edge applications, the **Evo family** (7B-40B) pushes the scale boundary but requires serious compute.
* Choose your model based on your task, sequence length, species scope, and available hardware.

**This notebook pairs with NB1-NB4 in the GLM module of BMI 511.**